# ADT — Abstract Data Types (9569)

Revision / content-lookup notebook for **nodes, linked lists, binary search trees, and hash tables**.

## Contents
1. [Node objects](#nodes) — OOP building block
2. [Linked list](#linked-list) — linear chain ADT
3. [Binary search tree](#bst) — hierarchical node ADT
4. [Hash table](#hash) — separate chaining with linked lists
5. [Complexity lookup](#complexity)
6. [Exam cheat-sheet](#cheatsheet)

**How to use:** skim markdown for the idea and stopping conditions; run code for exam-style patterns. Two representations appear in papers: **OOP + `None`** (this notebook) and **array of nodes + index pointers + free list** (theory/static storage).


<a id="nodes"></a>
## Node objects

A node is the basic storage unit: **data** plus **pointer(s)** to other node(s).

In a singly linked list, each node points to the **next** node (`None` at the end). In a tree, a node may point **left** and **right**.

**Exam habits:**
- encapsulate with `__data`, `__pointer` / `__left`, `__right`
- expose getters/setters
- `None` marks “no link” in the dynamic OOP version
- in array versions, `0` or `-1` often plays the role of `None`


In [23]:
class Node:
    def __init__(self, data=None):
        self.__data = data
        self.__right = None

    def get_data(self):
        return self.__data

    def set_data(self, data):
        self.__data = data

    def get_right(self):
        return self.__right

    def set_right(self, right):
        self.__right = right


<a id="linked-list"></a>
## Linked list

A linked list is a linear chain of node objects. The list object stores only the **head/start pointer**.

**Core operations:**
- `insert_end` / `insert_front` — add a new node
- `search` — traverse until match or `None`
- `delete` — handle head delete, middle delete, and not-found separately
- `display` / `to_list` — walk from head to tail

**Traversal pattern:** keep a `current` pointer; hop with `current = current.get_right()` until `current is None`.

**Delete pattern:** if head matches, move head forward; otherwise keep `previous` and `current`, then link `previous` to `current.get_right()` when found.


In [24]:
class LinkedList:
    def __init__(self, start=None):
        self.__start = start  # head pointer (Node object or None)

    def is_empty(self):
        return self.__start is None

    def insert_end(self, data):
        new = Node()
        new.set_data(data)
        if self.is_empty():
            self.__start = new
        else:
            current = self.__start
            while current.get_right() is not None:
                current = current.get_right()
            current.set_right(new)

    def insert_front(self, data):
        new = Node()
        new.set_data(data)
        if self.is_empty():
            self.__start = new
        else:
            new.set_right(self.__start)
            self.__start = new

    def search(self, data):
        # linear search by traversing the chain
        current = self.__start
        while current is not None and current.get_data() != data:
            current = current.get_right()
        return current  # Node if found, else None

    def delete(self, data):
        if self.is_empty():
            return
        if self.__start.get_data() == data:
            self.__start = self.__start.get_right()
            return

        previous = None
        current = self.__start
        while current is not None and current.get_data() != data:
            previous = current
            current = current.get_right()
        if current is None:
            return  # not found
        previous.set_right(current.get_right())

    def display(self):
        prtlist = ""
        current = self.__start
        while current:
            prtlist += f" -> {current.get_data()}"
            current = current.get_right()
        return prtlist


<a id="bst"></a>
## Binary search tree (BST)

TreeNode is still just a node: it stores data and links to other nodes. Here it adds a **left** pointer while the parent `Node` class already provides the **right** pointer via `get_right()` / `set_right()`.

The extra child pointer lets one node branch into **two ordered paths** instead of one linear next link. That enables ordered search similar in spirit to binary search: at each step you discard roughly half the remaining tree.

**Insert rule used here:** values `<=` current go **left**, values `>` go **right**.

**Memory hooks:**
- **Insert:** walk until an empty child slot, attach new node, `return`
- **Search:** same comparisons; stop on match or when you fall off the tree (`None`)
- **In-order:** left → root → right (prints ascending keys for a BST)

**Syllabus note:** BST **node deletion is excluded** from 9569. Focus on create/insert/search/traverse.


In [25]:
class TreeNode(Node):
    def __init__(self, data=None):
        super().__init__(data)
        self.__left = None

    def get_left(self):
        return self.__left

    def set_left(self, left):
        self.__left = left


class BST:
    def __init__(self, root=None):
        self.__root = root

    def get_root(self):
        return self.__root

    # insert rule: <= go left, > go right
    def insert(self, data):
        new = TreeNode(data)
        if self.__root is None:
            self.__root = new
            return

        current = self.__root
        while current:
            if data <= current.get_data():
                if current.get_left() is None:
                    current.set_left(new)
                    return
                current = current.get_left()
            else:
                if current.get_right() is None:
                    current.set_right(new)
                    return
                current = current.get_right()

    def search(self, data):
        current = self.__root
        while current is not None:
            if data == current.get_data():
                return current
            elif data < current.get_data():
                current = current.get_left()
            else:
                current = current.get_right()
        return None

    def in_order(self, node):
        if node is not None:
            self.in_order(node.get_left())
            print(node.get_data())
            self.in_order(node.get_right())


### Tree vs linked list (revision note)

TreeNode is just a node as it is a child of the Node class. It has similar behaviour and characteristics, with an additional left side pointer. The extra pointer allows for the node to point to 2 different paths, allowing one to traverse the graph following a logic akin to binary search. This allows for significantly faster traversals at O(log n) in a balanced tree, compared to O(n) of a linear linked list.


<a id="hash"></a>
## Hash table (separate chaining)

Create a fixed array of buckets. Use a hash function to map a key to a bucket index. Each bucket is a **linked list** for collision handling (**separate chaining**).

**Typical API:**
- `hash_function(key)` → index
- `insert(key)` → add to bucket chain (skip duplicates if already present)
- `search(key)` → check the bucket chain
- `delete(key)` → remove from the bucket chain
- `display()` → show each index and its chain

Average search is **O(1)** when keys spread evenly: hash is O(1) and the chain is short. If almost every key collides into one bucket, the chain becomes one long linked list and average search degrades toward **O(n)** due to linear traversal.

**Alternative (know for theory):** linear probing stores keys directly in the array and scans forward on collision. Chaining reuses linked-list code and is often easier under exam time.


In [26]:
class HashTable:
    def __init__(self, N):
        self.__n = N
        self.__arr = [LinkedList() for _ in range(N)]

    def hash_function(self, key):
        return key % self.__n

    def insert(self, key):
        bucket = self.__arr[self.hash_function(key)]
        if bucket.search(key) is None:
            bucket.insert_front(key)

    def search(self, key):
        return self.__arr[self.hash_function(key)].search(key) is not None

    def delete(self, key):
        self.__arr[self.hash_function(key)].delete(key)

    def display(self):
        for i, bucket in enumerate(self.__arr):
            chain = bucket.display()
            print(f"[{i}]{chain if chain else ''}")


<a id="complexity"></a>
## Complexity lookup

| ADT / operation | Average | Worst | Notes |
|---|---:|---:|---|
| Linked list search | O(n) | O(n) | linear traversal |
| Linked list insert at end | O(n) | O(n) | must walk to tail unless you keep a tail pointer |
| Linked list insert at front | O(1) | O(1) | |
| BST search | O(log n) | O(n) | worst if tree becomes a chain |
| BST insert | O(log n) | O(n) | same as search walk |
| Hash search (chaining) | O(1) | O(n) | worst when all keys share one bucket |
| In-order traversal | O(n) | O(n) | visits every node once |

### Exam distinctions
- **Dynamic OOP version:** objects + `None` terminators
- **Static array version:** node array + integer pointers + optional free list
- **BST delete:** out of syllabus; do not burn time memorising 3-case deletion
- **Hash collision handling:** separate chaining (this notebook) vs linear probing (theory)


<a id="cheatsheet"></a>
## Exam cheat-sheet

### Linked list
```text
new node → set data
empty? head = new
else walk to tail → tail.set_right(new)

search: current = head; while current and data mismatch → hop
delete head: head = head.next
delete else: previous/current walk; link previous to current.next
```

### BST
```text
rule: < or <= → left, else → right
insert: walk until empty child, attach, return
search: walk until match or None
in-order(node): if node: in_order(left); visit; in_order(right)
```

### Hash table (chaining)
```text
index = key % N
bucket[index] is a linked list
insert/search/delete → operate on that list only
display each [index] chain
```

<a id="tests"></a>
### Test scaffold


In [27]:
# test scaffold — run definition cells above first
print("=== linked list ===")
ll = LinkedList()
for v in [50, 20, 70, 10, 40]:
    ll.insert_end(v)
print(ll.display())
print("search 40:", ll.search(40) is not None)
print("search 99:", ll.search(99) is not None)
ll.delete(20)
print("after delete 20:", ll.display())

print("\n=== BST ===")
bst = BST()
for v in [50, 20, 70, 10, 40]:
    bst.insert(v)
print("search 40:", bst.search(40) is not None)
print("search 99:", bst.search(99) is not None)
print("in-order:")
bst.in_order(bst.get_root())

print("\n=== hash table (N=7) ===")
ht = HashTable(7)
for v in [50, 20, 70, 10, 40, 57, 15]:
    ht.insert(v)
ht.display()
print("search 40:", ht.search(40))
print("search 99:", ht.search(99))
ht.delete(20)
print("after delete 20:")
ht.display()


=== linked list ===
 -> 50 -> 20 -> 70 -> 10 -> 40
search 40: True
search 99: False
after delete 20:  -> 50 -> 70 -> 10 -> 40

=== BST ===
search 40: True
search 99: False
in-order:
10
20
40
50
70

=== hash table (N=7) ===
[0] -> 70
[1] -> 15 -> 57 -> 50
[2]
[3] -> 10
[4]
[5] -> 40
[6] -> 20
search 40: True
search 99: False
after delete 20:
[0] -> 70
[1] -> 15 -> 57 -> 50
[2]
[3] -> 10
[4]
[5] -> 40
[6]
